#Important Installs

In [1]:
!pip install evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=c014a95448a4844addc1c1e7bcc7988b897eae59839fa8cc408e00a5779a05d2
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


#Imports

In [2]:
import tarfile
import os
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import torch
import torch.nn as nn
from torch.optim import AdamW, Adam
from datasets import load_dataset
from transformers import BartForConditionalGeneration, BartTokenizer

import evaluate
from tqdm import tqdm
import nltk
nltk.download('wordnet')
nltk.download('punkt')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

# Universal Variable

For reproducability

## Setting up Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Set device to cuda if available

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setting Random Seed

In [5]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

## Links/ Paths

In [6]:
zippedDataSetPath = '/content/drive/MyDrive/Natural_Language_Processing/Dataset/english_XLSum_v2.0.tar.bz2'
extractedDataSetPath = '/content/xlsum_en_dataset'

modelSavePath = '/content/drive/MyDrive/Natural_Language_Processing/Model_Saves'

## Hyperparameters

In [7]:
MAX_SUMMARY_LENGTH = 64
MAX_INPUT_LENGTH = 512

BEAM_SEARCH_WIDTH = 4

BATCH_SIZE = 256
EPOCHS = 1

LEARNING_RATE = 1e-5

# Extracting and Loading Dataset

In [8]:
# Create Directory if it does not exist
os.makedirs(extractedDataSetPath, exist_ok=True)

# Extract the tar file
with tarfile.open(zippedDataSetPath, 'r:bz2') as tar:
    tar.extractall(path=extractedDataSetPath)

print("Files have been successfully extracted.")

/tmp/ipykernel_708/1810096438.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extractedDataSetPath)


Files have been successfully extracted.


In [9]:
xlsumEnDataset = load_dataset('json', data_files={
    'test': os.path.join(extractedDataSetPath, 'english_test.jsonl'),
    'val': os.path.join(extractedDataSetPath, 'english_val.jsonl')
})

Generating test split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [10]:
print(xlsumEnDataset['test'][0]['title'])
print(xlsumEnDataset['test'][0]['summary'])
print(xlsumEnDataset['test'][0]['text'])

Trump inauguration speech: 'Angry', 'authentic', 'primal'
Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.
By Kate DaileyBBC News Earlier this week, Trump posted a photo of himself sitting at a desk at Mar-a-Largo, a permanent marker hovering over a notepad. "Writing my inaugural address at the Winter White House, Mar-a-Lago, three weeks ago. Looking forward to Friday," he tweeted. Trump vows to end 'American carnage' Trump's angry call to arms Full text of Trump's inauguration speech It's unclear whether the president-elect actually wrote the speech himself, but the content was pure Trump: the same populist message that resonated throughout the primaries and the campaign. "Today, we are not merely transferring power from one administration to another, or from one party to another, but we are transferring power from Washington, DC, and giving it back to you, the people," he said at the beginnin

# Initializing tokenizer and Model

In [11]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

# Since we are generating text for summerization, we shall be using the BartForConditionalGenration specifically
model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

## Defining tokenize function for dataset tokenization

In [12]:
def tokenize(batch):
  # We shall truncate the input text to Bart's max length, which is 1024
  modelInput = tokenizer(batch['text'],padding='max_length',truncation=True,max_length=MAX_INPUT_LENGTH)
  # And truncate the summary to a length of 128
  labels = tokenizer(batch['summary'],padding='max_length',truncation=True,max_length=MAX_SUMMARY_LENGTH)

  modelInput['labels'] = labels['input_ids']
  return modelInput


## Actual mapping of tokenize function

In [13]:
testDs = xlsumEnDataset['test'].map(tokenize,batched=True)
valDs = xlsumEnDataset['val'].map(tokenize,batched=True)

Map:   0%|          | 0/11535 [00:00<?, ? examples/s]

Map:   0%|          | 0/11535 [00:00<?, ? examples/s]

In [14]:
testDs.set_format('torch',columns=['input_ids','attention_mask','labels'])
valDs.set_format('torch',columns=['input_ids','attention_mask','labels'])

# Evaluate Model Function

In [15]:
def evaluate_model(model,testLoader,tokenizer, device):
  model.eval()
  rougeMetric = evaluate.load('rouge')
  bleuMetric = evaluate.load('bleu')
  meteorMetric = evaluate.load('meteor')

  generatedSummary = []
  originalSummary = []

  # setting torch to not use gradients
  with torch.no_grad():
    for batch in tqdm(testLoader, desc="Evaluating"):
      inputId = batch['input_ids'].to(device)
      attentionMask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      # replace errors for padding
      labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)

      # when generating summary, we set the max length of summary the same as initilised and bean search width to be 4 and we set early stopping to be true incase there is no improvement
      generatedId = model.generate(input_ids = inputId, attention_mask = attentionMask, max_length=MAX_SUMMARY_LENGTH, num_beams=BEAM_SEARCH_WIDTH, early_stopping=True)

      # decoding the original and generated inputIds
      decodedGeneratedSummary = tokenizer.batch_decode(generatedId,skip_special_tokens=True)
      decodedOriginalSummary = tokenizer.batch_decode(labels,skip_special_tokens=True)

      # Filter out any None values or empty strings to prevent errors in the evaluate library
      decodedGeneratedSummary = [s for s in decodedGeneratedSummary if s is not None and s.strip() != '']
      decodedOriginalSummary = [s for s in decodedOriginalSummary if s is not None and s.strip() != '']

      generatedSummary.extend(decodedGeneratedSummary)
      originalSummary.extend(decodedOriginalSummary)


  # Computing rouge scores, while considering words with the same stem to be equal
  rougeResult = rougeMetric.compute(predictions=generatedSummary, references=originalSummary, use_stemmer=True)
  bleuResult = bleuMetric.compute(predictions=generatedSummary, references=originalSummary)
  meteorResult = meteorMetric.compute(predictions=generatedSummary, references=originalSummary)

  # Combine all results
  fullResult = {}
  fullResult.update(rougeResult)
  fullResult.update(bleuResult)
  fullResult.update(meteorResult)

  return fullResult

# Loading model, weights and dataset

In [16]:
torch.cuda.empty_cache()

## Loading Dataset

In [17]:
testLoader = torch.utils.data.DataLoader(testDs, batch_size=BATCH_SIZE, shuffle=False)
valLoader = torch.utils.data.DataLoader(valDs, batch_size=BATCH_SIZE, shuffle=False)

# Evaluating Models

In [19]:
model.to(device)

performanceMetrics = evaluate_model(model,testLoader,tokenizer, device)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
Evaluating: 100%|██████████| 46/46 [12:25<00:00, 16.21s/it]


In [20]:
print(performanceMetrics)

{'rouge1': np.float64(0.20276504297565184), 'rouge2': np.float64(0.027717621433389086), 'rougeL': np.float64(0.1305129718807297), 'rougeLsum': np.float64(0.13046163868154353), 'bleu': 0.008023383429173143, 'precisions': [0.13928416461766113, 0.015109320760582353, 0.002862578102184964, 0.0006879013458646869], 'brevity_penalty': 1.0, 'length_ratio': 2.3440432769148805, 'translation_length': 629167, 'reference_length': 268411, 'meteor': np.float64(0.20560056076711408)}
